<a href="https://colab.research.google.com/github/Sunidhishree/flyrank-ml-internship1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Sunidhishree/flyrank-ml-internship1"
REPO_DIR = "flyrank-ml-internship1"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Source: FlyRank, "The State of AI-Driven SEO," March 2026 data report (341,701 content pieces, 57 brands).

Finding 1 — ML Appendix, "What Predicts Health?" (Random Forest feature importance)

The paper trains a Random Forest to predict Health Score, and finds Average Position (43% importance), Impressions (32%), and Scroll Depth (15%) as the top predictors. But Health Score is explicitly defined earlier in the paper as a weighted sum: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). The paper itself flags this honestly — noting the target is "partly constructed from some of these inputs," so importance is descriptive, not causal.

My methodology question: Where exactly does the label come from, relative to the features? If Health Score is a deterministic formula built directly from Position, Impressions, and Scroll Depth, then a model "discovering" those same three features as top predictors isn't really discovering anything — it's re-deriving the formula it was trained to approximate. This is structurally the same leakage pattern I found in my own Week 3 notebook, where trend_direction turned out to be a deterministic bucketing of trend_pct. I'd ask: would this section be stronger framed not as "feature importance" at all, but explicitly as "recovering the known Health Score formula," since that's what the result actually shows?

Finding 2 — ML Appendix, "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

This model separates growing vs. declining pages using an 80/20 holdout split, with Content Age, Days Since Update, and Days Visible as the strongest signals.

My methodology question: Does the validation design carry this claim? The methodology section states the dataset spans 57 brands, but doesn't say whether the 80/20 split was random-by-row or grouped-by-brand. Given my own Week 4 experience — where a plain random split looked fine (0.652 accuracy) but a client-grouped split actually underperformed a naive baseline (0.612 vs. a 0.677 base rate) — I'd genuinely want to know whether this 71% holdout accuracy holds up on brands the model never saw during training, or whether some of that signal is really "this model learned brand-level quirks," not a general growth pattern. This isn't a criticism of the finding itself — the paper is transparent about being exploratory and descriptive — just a concrete, checkable question I'd want answered before trusting the number as portfolio-wide.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
feat_cols = ["impressions_90d", "ctr", "avg_position", "days_since_last_update"]
clean = df[df["avg_position"] > 0].dropna(subset=feat_cols).copy()

# BEFORE: random split (what w05 did)
scaler = StandardScaler()
X_all = scaler.fit_transform(clean[feat_cols])
km_random = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_random = km_random.fit_predict(X_all)
sil_random = silhouette_score(X_all, labels_random)
print("BEFORE (random split, no grouping):", round(sil_random, 3))

# AFTER: grouped split by client — fit on one set of clients, score on entirely unseen clients
gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
train_idx, test_idx = next(gss.split(clean, groups=clean["client_id"]))
train_g, test_g = clean.iloc[train_idx].copy(), clean.iloc[test_idx].copy()

X_train_g = scaler.fit_transform(train_g[feat_cols])
X_test_g = scaler.transform(test_g[feat_cols])
km_grouped = KMeans(n_clusters=4, random_state=42, n_init=10)
km_grouped.fit(X_train_g)
test_g["cluster"] = km_grouped.predict(X_test_g)

sil_test_g = silhouette_score(X_test_g, test_g["cluster"])
print("AFTER (grouped by client, scored on unseen clients):", round(sil_test_g, 3))

BEFORE (random split, no grouping): 0.511
AFTER (grouped by client, scored on unseen clients): 0.56


Before/after comparison: Random-split silhouette was 0.511. Grouped-split silhouette, scored on entirely unseen clients, was 0.560 — actually higher than the random split, not lower.

This is worth being honest about rather than forcing it to match my prediction: my hypothesis, based on my Week 4 warehouse notebook (where a grouped split revealed worse generalization than a random split), was that clustering would show the same weakness here. It didn't. The clusters appear to generalize well to clients the model never saw during fitting — if anything, the held-out grouped test set separated slightly more cleanly than the training-set random split did.

One honest caveat: this doesn't fully resolve the concern from my Finding 2 methodology question about the paper's logistic regression. My clustering task and the paper's growth-classification task are different problems (unsupervised structure vs. supervised prediction), so a stable result here doesn't guarantee the paper's classifier would show the same stability — it just means my specific method, on my specific features, doesn't show the client-overfitting pattern I was worried about. This is a genuinely positive, observed result for this model, not a general claim that clustering always generalizes well.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

corrs = df[feat_cols + ["is_declining_label"]].corr(numeric_only=True)["is_declining_label"].drop("is_declining_label")
print("Correlation of final feature set with label-derived column:")
print(corrs.sort_values(key=abs, ascending=False))

print("\nFinal feature set used in clustering:", feat_cols)
print("trend_pct or trend_direction present in feature set?",
      any(c in feat_cols for c in ["trend_pct", "trend_direction"]))

Correlation of final feature set with label-derived column:
days_since_last_update    0.081383
ctr                      -0.061911
avg_position             -0.029035
impressions_90d          -0.018175
Name: is_declining_label, dtype: float64

Final feature set used in clustering: ['impressions_90d', 'ctr', 'avg_position', 'days_since_last_update']
trend_pct or trend_direction present in feature set? False


Correlation of the four final clustering features with the label-derived decline indicator: days_since_last_update (+0.081), ctr (−0.062), avg_position (−0.029), impressions_90d (−0.018). All four are weak, well below any threshold that would suggest disguised label leakage. Confirmed: neither trend_pct nor trend_direction appears in the final feature set — only used earlier as a gate condition in the baseline rule (w04), never as a clustering input. This re-confirms the same conclusion from my Week 3 leakage check, now verified specifically against the exact four features used in the final k=4 model.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (too bold): "All 8 of the baseline's REFRESH-flagged pages fall inside Cluster 1, never Cluster 0 or Cluster 2 — a real confirmation that this part of the rule tracks genuine behavior."

Rewritten (safe language): "In this dataset and time window, all 8 pages flagged REFRESH by the baseline rule were observed to fall within a single cluster. This is a directional pattern consistent with the idea that the rule's staleness-plus-traffic logic tracks a real, distinct behavioral group — but it is decision-support evidence from one sample, not proof the relationship holds outside this data, and not a causal claim that staleness causes the REFRESH-worthy behavior."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.